# Лабораторная работа: выявление и устранение ошибок программного кода информационных систем

## 1. Контекст работы

Вам передан учебный прототип **системы обработки SQL-подобных запросов** к таблице пользователей. Программа запускается и может возвращать результаты запросов, однако пользователи сообщили о нескольких подозрительных ситуациях. Ваша задача — проверить каждое обращение, установить, действительно ли оно связано с дефектом, найти причину подтверждённых ошибок и устранить их. (**не изменяя реализацию бинарного дерева поиска (BST)**. С ним ничего не связано)


## 2. Как устроена система

### 2.1. Таблица и запросы

Данные хранятся в списке `users`. Каждая строка имеет три поля: `id` (числовой идентификатор), `name` (имя), `age` (возраст). Список `columns = ['id', 'name', 'age']` задаёт порядок полей. Например, `[1, 'Anna', 18]` — пользователь Anna, 18 лет.

Поддерживается **только** упрощённый формат запроса:

```sql
select name from users where age >= 18
```

Можно менять оператор сравнения (`=`, `>`, `<`, `>=`, `<=`) и целочисленное значение возраста. Выбирается только поле `name`, таблица всегда `users`, условие всегда по `age`. Это **не полноценный SQL-интерпретатор**: `AND`, `OR`, `ORDER BY`, произвольные имена таблиц и другие выражения не поддерживаются.

### 2.2. Способы поиска

- **SeqScan:** последовательный просмотр всех строк; условие проверяется для каждой строки.
- **HashTable:** поиск по хеш-индексу с методом цепочек. В корзине могут находиться разные ключи с одинаковым остатком от хеширования. Используется для поиска по равенству.
- **BST:** бинарное дерево поиска по возрасту. Узлы хранят возраст и идентификаторы строк; поддерживаются равенство и диапазонные сравнения. **Реализацию BST менять запрещено.**

Индексы хранят ссылки на строки в виде их позиций в списке. После изменения таблицы необходимо учитывать согласованность индекса с данными.

### 2.3. Планировщик и статистика

`analyze(table)` собирает статистику по возрастам в словаре `stats`: `age_counts` — сколько раз встречается каждый возраст, `analyzed_rows` — сколько строк было на момент анализа. **Добавление строк само по себе не обязано обновлять `stats`**: по условиям модели статистика пересчитывается при отдельном вызове `analyze()`.

`planner(N, op, value)` оценивает стоимость доступных способов выполнения и выбирает план. В учебной модели используются условные единицы затрат:

- `cost_SeqScan = N`;
- `cost_Hash = 1 + 4 * 2`;
- `cost_BST = log2(max(N, 1)) * X * 4`, где `X` берётся из статистики для запрошенного возраста.

Для диапазонных запросов рассматриваются `SeqScan` и `BST`, для равенства — `SeqScan` и `HashTable`.

### 2.4. Порядок выполнения запроса

1. `run_query()` разбирает текст запроса и извлекает оператор и возраст.
2. `planner()` выбирает алгоритм на основе размера таблицы и статистики.
3. Вызывается соответствующая функция поиска.
4. Из найденных строк извлекаются имена; программа печатает запрос, план, оценочную стоимость, `X`, `N` и результат.

## 3. Справочник функций

| Функция или объект | Назначение |
|---|---|
| `users`, `columns` | Таблица пользователей и порядок её полей |
| `column_index(column)` | Возвращает позицию столбца по имени |
| `compare(a, op, b)` | Сравнение двух значений |
| `seq_search(table, column, op, value)` | Последовательный поиск строк по условию |
| `build_hash_index(table, column, bucket_count)` | Строит хеш-индекс: корзины → ключи → позиции строк |
| `hash_search(table, index, value)` | Ищет строки по равенству через хеш-индекс |
| `BSTNode` | Класс узла бинарного дерева |
| `_bst_insert(node, key, row_id)` | Добавляет значение или позицию строки в BST |
| `build_bst_index(table, column)` | Создаёт BST-индекс по выбранному столбцу |
| `bst_search(table, root, op, value)` | Ищет строки по условию через BST |
| `stats` | Сохранённая статистика после последнего анализа |
| `analyze(table)` | Явно пересчитывает статистику по возрастам |
| `planner(N, op, value)` | Рассчитывает условную стоимость и выбирает способ поиска |
| `age_hash_index` | Сохранённый хеш-индекс по полю `age` |
| `run_query(query_string)` | Разбирает запрос, запускает выбранный поиск и выводит результат |
| `INITIAL_USERS` | Эталонный начальный набор строк для тест-кейсов |
| `reset_scenario(add_new=False)` | Восстанавливает таблицу и служебное состояние перед тестом; при `True` добавляет 15 пользователей |

## 4. Что разрешено и запрещено изменять

**Разрешено:** исправлять код сравнения, последовательного поиска, хеширования, поддержания актуальности индексов, анализа статистики, планировщика и выполнения запроса, **если вы обосновали необходимость исправления**. Можно добавлять диагностические `print`, временные проверки и собственные тесты.

**Запрещено:** изменять `BSTNode`, `_bst_insert`, `build_bst_index`, `bst_search`; удалять обращения пользователей; менять тестовые данные, условия и ожидаемый смысл запросов ради прохождения тестов; заменять всю систему заранее подготовленными ответами; исправлять поведение, которое соответствует требованиям.

**Служебные ячейки** с готовыми тест-кейсами предназначены для воспроизведения ситуаций и **не должны редактироваться**. Если нужен дополнительный тест, создайте новую ячейку.

## 5. Порядок выполнения лабораторной

1. Выполните основную ячейку с реализацией системы и служебную ячейку подготовки сценариев.
2. Запускайте тест-кейсы по одному. Каждый тест начинает работу с восстановленного набора данных.
3. Для каждого обращения определите ожидаемое поведение и сравните его с фактическим.
4. Если ошибка подтверждается, найдите место возникновения и объясните причину. Не изменяйте код до того, как сможете воспроизвести проблему.
5. Исправьте обнаруженный дефект в разрешённой части программы.
6. **Повторно выполните основную ячейку**, затем служебную ячейку и нужный тест-кейс (в Jupyter ранее созданные функции не обновляются автоматически после редактирования текста)
7. После исправлений проверьте все обращения ещё раз. Помните, что среди них могут быть сообщения о корректном поведении системы.


---
## 6. Исходный код системы

Следующая ячейка содержит реализацию. Ниже расположены служебная подготовка и обращения пользователей. В самом низу - обращения


## примечание

Данная работа по структуре очень похожа на то, с чем вы работали до этого, за исключением того, что теперь в ней могут присутствовать ошибки.

In [ ]:
users = [[1, 'Anna', 18],
        [2, 'Ivan', 21],
        [3, 'Petr', 17],
        [4, 'Maria', 25],
        [5, 'Alex', 18],
        [6, 'Olga', 30],
        [7, 'Nikita', 21]]

columns = ['id', 'name', 'age']


def column_index(column):
    return columns.index(column)


def compare(a, op, b):
    if op == '=':
        return a == b
    if op == '>':
        return a > b
    if op == '<':
        return a < b
    if op == '>=':
        return a > b
    if op == '<=':
        return a <= b
    raise ValueError(f'Неизвестный оператор: {op}')


#
# Блок с реализацией последовательного поиска
#

def seq_search(table, column, op, value):
    c = column_index(column)
    result = []

    for row in table:
        if compare(row[c], op, value):
            result.append(row)

    return result


#
#  Блок с реализацией хэширования
#
def build_hash_index(table, column, bucket_count=10):
    c = column_index(column)
    buckets = [[] for _ in range(bucket_count)]

    for row_id, row in enumerate(table):
        key = row[c]
        bucket_id = hash(key) % bucket_count
        bucket = buckets[bucket_id]

        found = False
        for item in bucket:
            if item[0] == key:
                item[1].append(row_id)
                found = True
                break

        if not found:
            bucket.append([key, [row_id]])

    return buckets


def hash_search(table, index, value):
    bucket_count = len(index)
    bucket_id = hash(value) % bucket_count
    bucket = index[bucket_id]

    for key, row_ids in bucket:
        if key != value:
            return []
        if key == value:
            return [table[row_id] for row_id in row_ids]

    return []



#
# Блок с реализацией BST
#

class BSTNode:
    def __init__(self, key, row_id):
        self.key = key
        self.row_ids = [row_id]
        self.left = None
        self.right = None


def _bst_insert(node, key, row_id):
    if node is None:
        return BSTNode(key, row_id)

    if key < node.key:
        node.left = _bst_insert(node.left, key, row_id)
    elif key > node.key:
        node.right = _bst_insert(node.right, key, row_id)
    else:
        node.row_ids.append(row_id)

    return node


def build_bst_index(table, column):
    c = column_index(column)
    root = None

    for row_id, row in enumerate(table):
        root = _bst_insert(root, row[c], row_id)

    return root


def bst_search(table, root, op, value):
    row_ids = []

    def walk(node):
        if node is None:
            return

        if op == '=':
            if value < node.key:
                walk(node.left)
            elif value > node.key:
                walk(node.right)
            else:
                row_ids.extend(node.row_ids)
            return

        if op == '>':
            if node.key > value:
                walk(node.left)
                row_ids.extend(node.row_ids)
                walk(node.right)
            else:
                walk(node.right)
            return

        if op == '>=':
            if node.key >= value:
                walk(node.left)
                row_ids.extend(node.row_ids)
                walk(node.right)
            else:
                walk(node.right)
            return

        if op == '<':
            if node.key < value:
                walk(node.left)
                row_ids.extend(node.row_ids)
                walk(node.right)
            else:
                walk(node.left)
            return

        if op == '<=':
            if node.key <= value:
                walk(node.left)
                row_ids.extend(node.row_ids)
                walk(node.right)
            else:
                walk(node.left)
            return

        raise ValueError(f'Неизвестный оператор: {op}')

    walk(root)
    row_ids.sort()
    return [table[row_id] for row_id in row_ids]


#
# Блок с планировщиком и анализом
#
import math

stats = {}

def analyze(table):
    counts = {}
    for row in table:
        age = row[column_index('age')]
        counts[age] = counts.get(age, 0) + 1
    stats['age_counts'] = counts
    stats['analyzed_rows'] = len(table)
    print('Статистика обновлена:', stats)


def planner(N, op, value):
    if N <= 0:
        return [0, 'SeqScan']
    x = stats.get('age_counts', {}).get(value, 0)
    cost_SeqScan = N
    cost_Hash = 1 + 4 * 2
    cost_BST = math.log2(max(N, 1)) * x * 4
    if op != '=':
        if cost_BST <= cost_SeqScan:
            return [cost_BST, 'BST']
        return [cost_SeqScan, 'SeqScan']
    if cost_Hash <= cost_SeqScan:
        return [cost_Hash, 'HashTable']
    return [cost_SeqScan, 'SeqScan']
#
#
#


age_hash_index = build_hash_index(users, 'age', bucket_count=5)

def run_query(query_string):
    query = query_string.lower().strip().split()

    if (len(query) != 8 or query[:4] != ['select', 'name', 'from', 'users']) or (query[4] != 'where') or (query[5] != 'age') or (query[6] not in ('=', '>', '<', '>=', '<=')):
        raise ValueError('Ожидается: select name from users where age = 18')

    value = int(query[7])
    op = query[6]
    plan = planner(len(users[0]) if op != '=' else len(users), op, value)

    if plan[1] == 'SeqScan':
        rows = seq_search(users, 'age', op, value)
    elif plan[1] == 'HashTable':
        rows = hash_search(users, age_hash_index, value)
    else:
        idx = build_bst_index(users, 'age')
        rows = bst_search(users, idx, op, value)

    names = [r[column_index('name')] for r in rows]
    print(f'Запрос: {query_string}')
    print(f'План: {plan[1]}, стоимость: {plan[0]:.3f}, X={stats.get("age_counts", {}).get(value, 0)}, N={len(users)}')
    print('Результат:', names)
    return names

query_string = 'select name from users where age = 18'
print('ЭТАП 1. Исходная таблица')
analyze(users)
run_query(query_string)

print('\nЭТАП 2. Добавляем 15 строк; статистику НЕ обновляем')
for i in range(15):
    users.append([8+i, f'New_{i+1}', 18])
run_query(query_string)

print('\nЭТАП 3. Обновляем статистику')
analyze(users)
run_query(query_string)


ЭТАП 1. Исходная таблица
Статистика обновлена: {'age_counts': {18: 2, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 7}
Запрос: select name from users where age = 18
План: SeqScan, стоимость: 7.000, X=2, N=7
Результат: ['Anna', 'Alex']

ЭТАП 2. Добавляем 15 строк; статистику НЕ обновляем
Запрос: select name from users where age = 18
План: HashTable, стоимость: 9.000, X=2, N=22
Результат: ['Anna', 'Alex']

ЭТАП 3. Обновляем статистику
Статистика обновлена: {'age_counts': {18: 17, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 22}
Запрос: select name from users where age = 18
План: HashTable, стоимость: 9.000, X=17, N=22
Результат: ['Anna', 'Alex']


['Anna', 'Alex']



---



---



---



## **Обращения пользователей: тестовые сценарии**

Запускайте сценарии **по отдельности**, сверху вниз. Каждый сценарий сам восстанавливает исходные данные, поэтому результаты не зависят от предыдущего сценария.

В конце для каждого обращения укажите: «подтверждено / не подтверждено», наблюдаемый результат, причину, исправление и повторную проверку.


In [ ]:
INITIAL_USERS = [
    [1, 'Anna', 18], [2, 'Ivan', 21], [3, 'Petr', 17],
    [4, 'Maria', 25], [5, 'Alex', 18], [6, 'Olga', 30],
    [7, 'Nikita', 21],
]

def reset_scenario(add_new=False):
    global users, age_hash_index
    users = [row.copy() for row in INITIAL_USERS]
    stats.clear()
    analyze(users)
    age_hash_index = build_hash_index(users, 'age', bucket_count=5)
    if add_new:
        for i in range(15):
            users.append([8 + i, f'New_{i+1}', 18])

print('Сценарии готовы к запуску.')


Сценарии готовы к запуску.


### Обращение № 1
> «Мы загрузили ещё 15 пользователей, всем указали возраст 18 лет. Но поиск по возрасту 18, похоже, не показывает всех пользователей. Проверьте, пожалуйста».


In [ ]:
reset_scenario(add_new=True)
print('Число записей в таблице:', len(users))
run_query('select name from users where age = 18')

print()
print('Что реально есть в таблице:')
print(users)


Статистика обновлена: {'age_counts': {18: 2, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 7}
Число записей в таблице: 22
Запрос: select name from users where age = 18
План: HashTable, стоимость: 9.000, X=2, N=22
Результат: ['Anna', 'Alex']

Что реально есть в таблице:
[[1, 'Anna', 18], [2, 'Ivan', 21], [3, 'Petr', 17], [4, 'Maria', 25], [5, 'Alex', 18], [6, 'Olga', 30], [7, 'Nikita', 21], [8, 'New_1', 18], [9, 'New_2', 18], [10, 'New_3', 18], [11, 'New_4', 18], [12, 'New_5', 18], [13, 'New_6', 18], [14, 'New_7', 18], [15, 'New_8', 18], [16, 'New_9', 18], [17, 'New_10', 18], [18, 'New_11', 18], [19, 'New_12', 18], [20, 'New_13', 18], [21, 'New_14', 18], [22, 'New_15', 18]]


### Обращение № 2
> «В системе точно есть пользователь Olga, возраст 30. После расширения базы поиск по этому возрасту ничего не показывает».


In [ ]:
reset_scenario(add_new=True)
run_query('select name from users where age = 30')


Статистика обновлена: {'age_counts': {18: 2, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 7}
Запрос: select name from users where age = 30
План: HashTable, стоимость: 9.000, X=1, N=22
Результат: []


[]

### Обращение № 3
> «Я выбираю пользователей в возрасте 18 лет и старше, но некоторых совершеннолетних в выдаче нет».


In [ ]:
reset_scenario()
run_query('select name from users where age >= 18')


print()
print('Что реально есть в таблице:')
print(users)

Статистика обновлена: {'age_counts': {18: 2, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 7}
Запрос: select name from users where age >= 18
План: SeqScan, стоимость: 3.000, X=2, N=7
Результат: ['Ivan', 'Maria', 'Olga', 'Nikita']

Что реально есть в таблице:
[[1, 'Anna', 18], [2, 'Ivan', 21], [3, 'Petr', 17], [4, 'Maria', 25], [5, 'Alex', 18], [6, 'Olga', 30], [7, 'Nikita', 21]]


### Обращение № 4
> «После расширения базы планировщик, кажется, неправильно оценивает объём данных для диапазонного запроса. Проверьте расчёт стоимости и аргументы планировщика».

In [ ]:
reset_scenario(add_new=True)
print('Реальное количество строк:', len(users))
print('Количество столбцов:', len(users[0]))
run_query('select name from users where age > 18')


Статистика обновлена: {'age_counts': {18: 2, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 7}
Реальное количество строк: 22
Количество столбцов: 3
Запрос: select name from users where age > 18
План: SeqScan, стоимость: 3.000, X=2, N=22
Результат: ['Ivan', 'Maria', 'Olga', 'Nikita']


['Ivan', 'Maria', 'Olga', 'Nikita']

### Обращение № 5
> «После добавления пользователей статистика по возрасту осталась прежней. Наверное, система работает неправильно».

Проверьте утверждение с учётом правил обновления статистики в задании. Не считайте каждое отличие статистики от текущих данных дефектом автоматически.


In [ ]:
reset_scenario()
print('До загрузки:')
print('Статистика для age=18:', stats['age_counts'].get(18, 0))
for i in range(15):
    users.append([8 + i, f'New_{i+1}', 18])
print('После загрузки без обновления статистики:')
print('Статистика для age=18:', stats['age_counts'].get(18, 0))
print('Фактическое число строк с age=18:', sum(r[2] == 18 for r in users))
print('После явного обновления:')
analyze(users)
print('Статистика для age=18:', stats['age_counts'].get(18, 0))


Статистика обновлена: {'age_counts': {18: 2, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 7}
До загрузки:
Статистика для age=18: 2
После загрузки без обновления статистики:
Статистика для age=18: 2
Фактическое число строк с age=18: 17
После явного обновления:
Статистика обновлена: {'age_counts': {18: 17, 21: 2, 17: 1, 25: 1, 30: 1}, 'analyzed_rows': 22}
Статистика для age=18: 17
